In [13]:
!pip install -q timm

In [14]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms.v2 as v2
import timm
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
from pathlib import Path

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
model_dir = "convnextv2_tiny.fcmae_ft_in22k_in1k"
image_size = 384
num_aircraft_variants = 100

temp_model = timm.create_model(model_dir, pretrained=True)  # temp model
data_config = timm.data.resolve_model_data_config(
    temp_model
)  # Changed 'model' to 'temp_model'

In [25]:
train_transforms = v2.Compose([
    v2.Resize(size=[image_size,image_size]),
    v2.Pad(padding=0, padding_mode='edge'),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandAugment(num_ops=2, magnitude=9),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale = True),
    v2.Normalize(
        mean=data_config['mean'],
        std=data_config['std']
    ),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random')
])

val_transforms = v2.Compose([
    v2.Resize(size=[image_size,image_size]),
    v2.Pad(padding=0, padding_mode='edge'),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=data_config['mean'], std=data_config['std'])
])

#VERIFICATION OF MODEL SETUP
print(f"The model has loaded with {num_aircraft_variants} output classes, and target size of {image_size}X{image_size} ")

dataset_train = torchvision.datasets.FGVCAircraft(
    root='./data',
    split='train',
    download=True,
    annotation_level='variant',
    transform = train_transforms,
)

dataset_val = torchvision.datasets.FGVCAircraft(
    root='./data',
    split='val',
    download=True,
    annotation_level='variant',
    transform = val_transforms,
)

dataset_test = torchvision.datasets.FGVCAircraft(
    root='./data',
    split='test',
    download=True,
    annotation_level='variant',
    transform = val_transforms,
)



# Colab GPU Optimizations: num_workers=2 and pin_memory=True prevent CPU data bottlenecks
train_loader = DataLoader(dataset_train, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(dataset_test, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)


The model has loaded with 100 output classes, and target size of 384X384 


In [18]:
model = timm.create_model(
    model_dir,
    pretrained=True,
    num_classes=num_aircraft_variants,
    drop_path_rate=0.1
).to(device)

In [19]:

ds_fam = torchvision.datasets.FGVCAircraft(root='./data', split='train', annotation_level='family')
ds_manu = torchvision.datasets.FGVCAircraft(root='./data', split='train', annotation_level='manufacturer')

variant_to_family = {}
variant_to_manu = {}

for i in range(len(dataset_train)):
    variant_idx = dataset_train._labels[i]
    family_idx = ds_fam._labels[i]
    manu_idx = ds_manu._labels[i]

    variant_to_family[variant_idx] = family_idx
    variant_to_manu[variant_idx] = manu_idx

# Colab GPU Addition: Convert dictionaries to lookup tensors and push to CUDA device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
v2f_tensor = torch.tensor([variant_to_family[i] for i in range(num_aircraft_variants)], device=device)
v2m_tensor = torch.tensor([variant_to_manu[i] for i in range(num_aircraft_variants)], device=device)

print(f"Training set: {len(dataset_train)} samples across {num_aircraft_variants} Variants.")
print(f"Validation set: {len(dataset_val)} samples.")
print(f"Hierarchical maps generated: 100 Variants -> {len(set(variant_to_family.values()))} Families -> {len(set(variant_to_manu.values()))} Manufacturers.")

Training set: 3334 samples across 100 Variants.
Validation set: 3333 samples.
Hierarchical maps generated: 100 Variants -> 70 Families -> 30 Manufacturers.


In [20]:
for param in model.parameters():
    param.requires_grad = True

In [21]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params}")
print(f"Trainable params: {trainable_params}")

Total params: 27943396
Trainable params: 27943396


In [22]:
criterion = torch.nn.CrossEntropyLoss()

backbone_params = []
head_params = []

for name, param in model.named_parameters():
    if "head" in name or "fc" in name or "classifier" in name:
        head_params.append(param)
    else:
        backbone_params.append(param)

optimiser = optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 1e-4}
    ],
    weight_decay=0.05
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=10, eta_min=1e-6)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 10

# Network training

for epoch in range(num_epochs):
    print(f"Epoch number {epoch+1} of {num_epochs}")
    model.train()

    running_loss=0.0
    correct_preds=0
    total_samples=0

    for batch_idx, (images,targets) in enumerate(train_loader):
        # FIX FOR COLAB GPU: Send input batch to T4 GPU
        images, targets = images.to(device), targets.to(device)

        pred = model(images)
        loss = criterion(pred,targets)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        running_loss += loss.item() * images.size(0)
        preds = pred.argmax(dim=1)
        correct_preds += (preds == targets).sum().item()
        total_samples += targets.size(0)

    scheduler.step()

    epoch_loss = running_loss / total_samples
    epoch_acc = (correct_preds / total_samples) * 100
    print(f"Epoch {epoch+1}: Training loss of {epoch_loss:.4f} | Training accuracy of {epoch_acc:.2f}%")

print("Training complete")

Epoch number 1 of 10
Epoch 1: Training loss of 3.9498 | Training accuracy of 10.68%
Epoch number 2 of 10
Epoch 2: Training loss of 2.3384 | Training accuracy of 40.85%
Epoch number 3 of 10
Epoch 3: Training loss of 1.3370 | Training accuracy of 66.23%
Epoch number 4 of 10
Epoch 4: Training loss of 0.8213 | Training accuracy of 80.50%
Epoch number 5 of 10
Epoch 5: Training loss of 0.5541 | Training accuracy of 88.72%
Epoch number 6 of 10
Epoch 6: Training loss of 0.3917 | Training accuracy of 92.95%
Epoch number 7 of 10
Epoch 7: Training loss of 0.3047 | Training accuracy of 95.68%
Epoch number 8 of 10
Epoch 8: Training loss of 0.2503 | Training accuracy of 97.06%
Epoch number 9 of 10
Epoch 9: Training loss of 0.2213 | Training accuracy of 97.39%
Epoch number 10 of 10
Epoch 10: Training loss of 0.1943 | Training accuracy of 97.75%
Training complete


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# validation testing
model.eval()

variant_correct = 0
family_correct = 0
manu_correct = 0
total_val_samples = 0

# FIX 1 FOR COLAB GPU: Send lookup tensors to the GPU device
v2f_tensor = torch.tensor([variant_to_family[i] for i in range(len(variant_to_family))], device=device)
v2m_tensor = torch.tensor([variant_to_manu[i] for i in range(len(variant_to_manu))], device=device)

with torch.no_grad():
    for images, targets in val_loader:
        # FIX 2 FOR COLAB GPU: Send input batch to the GPU device
        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        variant_correct += (preds == targets).sum().item()

        pred_fam = v2f_tensor[preds]
        true_fam = v2f_tensor[targets]
        family_correct += (pred_fam == true_fam).sum().item()

        pred_manu = v2m_tensor[preds]
        true_manu = v2m_tensor[targets]
        manu_correct += (pred_manu == true_manu).sum().item()

        total_val_samples += targets.size(0)

variant_acc = (variant_correct/total_val_samples) * 100
family_acc = (family_correct/total_val_samples) * 100
manu_acc = (manu_correct/total_val_samples) * 100

print("Validation testing:")
print(f"Variant accuracy: {variant_acc:.2f}%")
print(f"Family accuracy: {family_acc:.2f}%")
print(f"Manufacturer accuracy: {manu_acc:.2f}%")

Validation testing:
Variant accuracy: 86.35%
Family accuracy: 92.83%
Manufacturer accuracy: 96.25%


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Final Test Dataset Evaluation
model.eval()

variant_correct = 0
family_correct = 0
manu_correct = 0
total_test_samples = 0

# Send lookup tensors to the GPU device
v2f_tensor = torch.tensor([variant_to_family[i] for i in range(len(variant_to_family))], device=device)
v2m_tensor = torch.tensor([variant_to_manu[i] for i in range(len(variant_to_manu))], device=device)

with torch.no_grad():
    for images, targets in test_loader:
        # Send input batch to the GPU device
        images, targets = images.to(device), targets.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        variant_correct += (preds == targets).sum().item()

        pred_fam = v2f_tensor[preds]
        true_fam = v2f_tensor[targets]
        family_correct += (pred_fam == true_fam).sum().item()

        pred_manu = v2m_tensor[preds]
        true_manu = v2m_tensor[targets]
        manu_correct += (pred_manu == true_manu).sum().item()

        total_test_samples += targets.size(0)

variant_acc = (variant_correct / total_test_samples) * 100
family_acc = (family_correct / total_test_samples) * 100
manu_acc = (manu_correct / total_test_samples) * 100

print("Final Test Testing:")
print(f"Variant accuracy: {variant_acc:.2f}%")
print(f"Family accuracy: {family_acc:.2f}%")
print(f"Manufacturer accuracy: {manu_acc:.2f}%")

Final Test Testing:
Variant accuracy: 87.61%
Family accuracy: 93.40%
Manufacturer accuracy: 96.34%
